## Text Encoding

## What is Text Encoding?

**Text encoding** (in the machine learning sense, not the character-encoding sense like UTF-8) is the process of converting raw, unstructured text — sentences, documents, reviews — into a **numeric representation** that a machine learning model can actually work with. Every model covered so far in this series (PCA, logistic regression, t-SNE, etc.) requires numeric input; text is not numeric on its own, so it must first be transformed into vectors of numbers before any of these techniques can be applied to it.

This is the same underlying problem solved earlier for categorical data (recall `DictVectorizer` turning `Type='h'` into a one-hot column) — text encoding is the analogous idea applied to *free-form written text* rather than a fixed set of category labels, which is a harder problem since text has no predefined set of possible values and can be arbitrarily long.

## What is a Corpus?

A **corpus** (plural: *corpora*) is simply the full collection of text documents being analyzed together — the entire dataset of text. Each individual document, sentence, or review within it is one "sample," and the corpus as a whole is what's used to build things like the vocabulary (see below): scanning across *every* document in the corpus to find every unique word that appears anywhere in it.

In the example given, the corpus consists of exactly two documents: `Doc1` and `Doc2`.

## What is Stemming / Lemmatization?

Both are preprocessing techniques used to reduce different grammatical forms of a word down to one common form, so the vocabulary doesn't end up treating closely related words (e.g. "run," "running," "ran") as entirely separate, unrelated terms.

- **Stemming** chops words down to a crude root form using simple, rule-based suffix-stripping — e.g. "programming" → "program", "loved" → "lov". It's fast, but the result isn't always a real word (as with "lov" above), since it doesn't understand grammar, just pattern-matches common endings.
- **Lemmatization** is a more linguistically-aware version of the same idea: it reduces a word to its dictionary base form (its *lemma*) using actual vocabulary and grammar rules — e.g. "better" → "good", "ran" → "run". It's slower than stemming, but produces cleaner, valid-word results.

Both serve the same underlying purpose flagged in the vocabulary-creation step: standardizing text (along with removing punctuation and lowercasing) shrinks the vocabulary size and stops the model from wastefully treating "Programming," "programming," and "programs" as three completely unrelated dimensions in the vector space, when they really carry the same core meaning.

Bag of Words (BoW):

The Bag of Words (BoW) model is one of the simplest methods of text encoding. Here's how it works:

- Vocabulary Creation:
    A vocabulary is created by listing all the unique words in the text corpus.
    Commonly, preprocessing steps like removing punctuation, converting to lowercase, and stemming/lemmatization are applied to standardize the text and reduce the vocabulary size.

- Text Vectorization:
    Each document/text is represented as a vector in a multi-dimensional space, where each dimension corresponds to a term (word) in the vocabulary.
    The value in each dimension is the frequency of that term in the document.

For example, consider two documents:

    Doc1: "I love programming."
    Doc2: "Programming is fun."

The vocabulary will be:

    ['I', 'love', 'programming', 'is', 'fun']

The BoW representations will be:

    BoW(Doc1) = [1, 1, 1, 0, 0]
    BoW(Doc2) = [0, 0, 1, 1, 1]

  ## Walking Through the BoW Example

**Step 1 — Vocabulary creation.** Every unique word across the *entire corpus* (both documents combined) is collected into one master list — the vocabulary:


Note this vocabulary has exactly 5 entries — one for every distinct word that appears *anywhere* across Doc1 and Doc2 combined (with "programming" counted only once, even though it appears in both documents). This fixed-length vocabulary list is what defines the number of dimensions every document's vector will have, going forward.

**Step 2 — Text vectorization.** Each document is now converted into a vector of length 5 (matching the vocabulary size), where **position *i* in the vector holds the count of how many times vocabulary word *i* appears in that document**:

| Vocabulary word | I | love | programming | is | fun |
|---|---|---|---|---|---|
| **BoW(Doc1)** — "I love programming." | 1 | 1 | 1 | 0 | 0 |
| **BoW(Doc2)** — "Programming is fun." | 0 | 0 | 1 | 1 | 1 |

Reading this row by row:
- **Doc1** contains "I" (count 1), "love" (count 1), and "programming" (count 1) — but never uses "is" or "fun," so those positions are 0. → `[1, 1, 1, 0, 0]`
- **Doc2** contains "programming" (count 1), "is" (count 1), and "fun" (count 1) — but never uses "I" or "love," so those positions are 0. → `[0, 0, 1, 1, 1]`

Notice the word **"programming" appears in both vectors** (position 3, value 1 in each) — this is the shared word between the two documents, and it's exactly why BoW vectors can be compared: two documents that share more vocabulary words in common will have vectors that are more similar to each other (e.g. via cosine similarity or Euclidean distance), which is the whole point of turning text into numbers in the first place — it makes "how similar are these two pieces of text" into a concrete, computable question.

**Key limitation worth noting** : this representation completely discards **word order** — "I love programming" and "programming love I" would produce the exact same BoW vector — and it treats every word as entirely independent, with no notion that "love" and "like" mean similar things. This is exactly why "bag" is in the name: it's as if all the words were dumped into a bag, counted, and the order/structure thrown away.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd
# CountVectorizer is scikit-learn's built-in tool for automatically
# doing exactly what was done by hand in the BoW example above —
# building the vocabulary AND vectorizing every document into a count
# vector, in one step, without manually listing unique words or
# counting occurrences yourself.
# TfidfVectorizer is imported alongside it, signaling the notebook
# will compare BoW against TF-IDF (a more refined encoding) shortly.
# pandas is used to display the resulting vectors as a readable table.

# Toy text data
documents = [
    'I love programming.',
    'Python is a versatile language.',
    'Data science is an interesting field.',
    'Machine learning is a subset of data science.'
]
# A small corpus of 4 documents (recall: "corpus" = the full collection
# of text documents being analyzed together) — larger and more varied
# than the 2-document Doc1/Doc2 example, so the resulting vocabulary
# and vectors will be more substantial.

# -------------------
# Bag of Words
# -------------------
vectorizer_bow = CountVectorizer()
# Create a CountVectorizer instance with default settings. By default,
# it automatically lowercases all text and strips punctuation before
# building the vocabulary — handling some of the "standardization"
# preprocessing mentioned earlier (though NOT stemming/lemmatization,
# which CountVectorizer doesn't do on its own; "programming" and
# "program" would still be treated as different words unless you
# preprocess the text yourself first or supply a custom tokenizer).

X_bow = vectorizer_bow.fit_transform(documents)
# In one call: scan all 4 documents to build the vocabulary (fit),
# then convert each document into its count vector (transform) — same
# fit_transform pattern used by PCA, t-SNE, DictVectorizer, etc.
# throughout this series. Returns a sparse matrix (mostly zeros, like
# the DictVectorizer output from the Melbourne housing notebook), since
# most documents only use a small fraction of the total vocabulary.

df_bow = pd.DataFrame(X_bow.toarray(), columns=vectorizer_bow.get_feature_names_out())
# Convert the sparse matrix to a dense array (.toarray()) and wrap it
# in a DataFrame for readability, using vectorizer_bow.get_feature_names_out()
# to label each column with the actual vocabulary word it represents —
# analogous to vec.get_feature_names_out() from DictVectorizer earlier.

df_bow
# Display the resulting table: 4 rows (one per document) x 15 columns
# (one per unique vocabulary word found across all 4 documents).

,an,data,field,interesting,is,language,learning,love,machine,of,programming,python,science,subset,versatile
0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0
1,0,0,0,0,1,1,0,0,0,0,0,1,0,0,1
2,1,1,1,1,1,0,0,0,0,0,0,0,1,0,0
3,0,1,0,0,1,0,1,0,1,1,0,0,1,1,0


## Actual Output

| | an | data | field | interesting | is | language | learning | love | machine | of | programming | python | science | subset | versatile |
|---|----|------|-------|-------------|----|----------|----------|------|---------|----|-------------|--------|---------|--------|-----------|
| **0** (I love programming.) | 0 | 0 | 0 | 0 | 0 | 0 | 0 | **1** | 0 | 0 | **1** | 0 | 0 | 0 | 0 |
| **1** (Python is a versatile language.) | 0 | 0 | 0 | 0 | **1** | **1** | 0 | 0 | 0 | 0 | 0 | **1** | 0 | 0 | **1** |
| **2** (Data science is an interesting field.) | **1** | **1** | **1** | **1** | **1** | 0 | 0 | 0 | 0 | 0 | 0 | 0 | **1** | 0 | 0 |
| **3** (Machine learning is a subset of data science.) | 0 | **1** | 0 | 0 | **1** | 0 | **1** | 0 | **1** | **1** | 0 | 0 | **1** | **1** | 0 |

A few things worth noticing in this real result:

- **Vocabulary size is 15**, alphabetically sorted by default (`get_feature_names_out()` returns them in that order, which is why the columns run `an, data, field, ...` rather than in the order words first appeared).
- **Single-letter words are gone**: notice "I" (from "I love programming") and "a" (from documents 1 and 3) don't appear anywhere in the vocabulary. This is CountVectorizer's default `token_pattern`, which only counts tokens of **2 or more characters** — silently dropping single-character "words" like "I" and "a" as likely not meaningful for BoW purposes. This is a common surprise the first time you compare CountVectorizer's automatic output against a hand-built vocabulary like an earlier example, which manually kept "I" as its own dimension.
- **"is" appears in documents 1, 2, and 3** but not document 0 — exactly the kind of very common, low-information word (a "stop word") that BoW's raw counts treat as equally important as distinctive words like "programming" or "science." This is typically the motivation for TF-IDF, which down-weights common words like "is" precisely because they appear everywhere and don't help distinguish one document from another — likely the point the upcoming `TfidfVectorizer` cell will make explicit.

# TF-IDF: Term Frequency – Inverse Document Frequency

## What is TF (Term Frequency)?

**Term Frequency (TF)** measures how often a term appears *within a single document*, exactly as described:

$$
TF(t) = \text{Number of times term } t \text{ appears in a document}
$$

This is the same quantity BoW already computes — the raw count in each cell of the BoW table from before. On its own, TF says nothing about whether a term is distinctive or just a common word — "is" and "programming" are both just counted, with no distinction between them.

## What is IDF (Inverse Document Frequency)?

**Inverse Document Frequency (IDF)** measures how *rare* or *distinctive* a term is across the whole corpus, not just one document:

$$
IDF(t) = \log\left(\frac{\text{Total number of documents}}{\text{Number of documents containing term } t}\right)
$$

The intuition is in the name — it's the *inverse* of document frequency: a term that appears in **many** documents gets a **small** IDF (log of a ratio close to 1, i.e. close to 0), while a term that appears in **few** documents gets a **large** IDF (log of a big ratio).

**Concretely, on the 4-document corpus from before:**

| Term | Appears in how many of the 4 docs? | Relative IDF |
|---|---|---|
| "is" | 3 of 4 docs | **Low** — very common, barely distinguishes anything |
| "data" / "science" | 2 of 4 docs | Medium |
| "programming" / "love" | 1 of 4 docs | **High** — rare, highly distinctive |

(Note: scikit-learn's `TfidfVectorizer` uses a slightly smoothed version of this formula by default — adding 1 to both the numerator and denominator counts, and adding 1 to the final result, to avoid division-by-zero and to prevent terms appearing in *every* document from getting an IDF of exactly 0. The ranking/intuition above still holds exactly.)

## TF-IDF Score

$$
TF\text{-}IDF(t) = TF(t) \times IDF(t)
$$

Multiplying the two together means a term only gets a **high TF-IDF score** if it satisfies *both* conditions at once: it must appear **often in this particular document** (high TF) **and** be **rare across the rest of the corpus** (high IDF). A term that's frequent in a document but also frequent everywhere else (like "is") gets dragged back down by its low IDF, no matter how many times it's repeated locally.

## Why Is This Necessary?

This directly fixes the exact weakness flagged with plain BoW: recall that "is" appeared in documents 1, 2, and 3 of the earlier example, getting a raw count of 1 in each — **numerically indistinguishable** from a rare, meaningful word like "programming," which also got a count of 1. BoW has no way to express "this word matters more than that one" — every word is weighted purely by how many times it's repeated, regardless of whether it's a near-meaningless connector word ("is," "the," "a") or a genuinely content-bearing word ("programming," "science").

TF-IDF exists specifically to correct this: it automatically **down-weights words that are common across the whole corpus** (since they carry little information about what makes any *one* document distinctive) and **up-weights words that are rare and concentrated in just a few documents** (since those are the words that actually characterize what a document is about).

This is visible directly in the real TF-IDF output for this corpus:

| | is | programming | data | science |
|---|---|---|---|---|
| Doc 0 (I love programming.) | 0.000 | **0.707** | 0.000 | 0.000 |
| Doc 1 (Python is a versatile language.) | 0.346 | 0.000 | 0.000 | 0.000 |
| Doc 2 (Data science is an interesting field.) | 0.296 | 0.000 | 0.366 | 0.366 |
| Doc 3 (Machine learning is a subset of data science.) | 0.269 | 0.000 | 0.332 | 0.332 |

Even though "is" appears (with count 1) in documents 1, 2, and 3, its TF-IDF score (0.346, 0.296, 0.269) stays consistently **lower** than "programming"'s score in document 0 (0.707) or "data"/"science"'s scores (0.366, 0.332) — despite all of these having the exact same raw count of 1 in BoW. The rarer, more distinctive words are correctly boosted above the common connector word.

## When Is TF-IDF Used?

TF-IDF is the standard choice whenever the goal is to capture **what makes each document distinctive**, rather than just its raw word content. Typical use cases:

- **Search engines / information retrieval** — ranking documents by relevance to a query, since query terms that are rare-but-present in a document are strong relevance signals.
- **Document similarity / clustering** — comparing documents by their TF-IDF vectors tends to group them by actual topic, since common filler words no longer dominate the similarity calculation.
- **Text classification** (e.g. spam detection, sentiment analysis) — as input features for models like logistic regression or SVMs, where distinctive words are far more useful predictors than ubiquitous ones.
- **Keyword extraction** — the highest TF-IDF terms in a document are often a good automatic summary of what that document is uniquely "about."

**When plain BoW might still be preferred instead:** if word frequency itself (not distinctiveness) is what matters — e.g. some topic-modeling algorithms (like LDA) are specifically built to work on raw counts — or when working with very short, near-identical documents where corpus-wide rarity isn't a meaningful signal.

As the original text notes, both representations — whether raw BoW counts or TF-IDF-weighted vectors — ultimately produce the same kind of thing: a numeric vector per document, ready to be fed as input into any downstream machine learning model (classification, clustering, dimensionality reduction, etc.), exactly the same way the Iris measurements or Melbourne housing features were.

In [ ]:
df_bow.values
# Access the underlying raw NumPy array stored inside the df_bow
# DataFrame, stripping away the row index and column labels
# (the vocabulary words) — returning just the plain 4x15 numeric
# array of counts.
#
# This is the reverse direction of what pd.DataFrame(X_bow.toarray(), ...)
# did earlier: that call took a plain array and wrapped it in a
# DataFrame (adding readable column names); .values unwraps it back
# to just the numbers.
#
# Useful whenever a downstream step needs a plain NumPy array rather
# than a DataFrame — e.g. feeding this directly into a scikit-learn
# model's .fit(), or into PCA/t-SNE as done with X earlier in this
# series, since those expect array-like numeric input and don't need
# (or use) the column name labels.

array([[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1],
       [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0]])

In [ ]:
(df_bow > 0).sum(axis=0)
# Two steps combined:
#
#   1. df_bow > 0
#      Compares every value in the DataFrame to 0, producing a new
#      DataFrame of the SAME shape (4 rows x 15 columns), but filled
#      with True/False instead of counts — True wherever a word
#      appeared at least once in that document (count > 0), False
#      wherever it didn't appear at all (count == 0).
#
#   2. .sum(axis=0)
#      Sums DOWN each column (axis=0 means "collapse the rows"),
#      adding up the True/False values per column. Since True behaves
#      as 1 and False as 0 in a sum, this counts, for EACH WORD, how
#      many of the 4 documents that word appeared in at least once.
#
# The result is a single row (a pandas Series) with one number per
# vocabulary word — this is precisely the DOCUMENT FREQUENCY of each
# term: "number of documents containing term t," the exact denominator
# used inside the IDF formula from before:
#
#   IDF(t) = log(Total number of documents / Number of documents with term t)
#
# So this line is effectively computing, by hand, the raw ingredient
# TfidfVectorizer uses internally to calculate each word's IDF score —
# a good way to verify or inspect those document-frequency counts
# directly, rather than trusting the TF-IDF numbers as a black box.

,0
an,1
data,2
field,1
interesting,1
is,3
language,1
learning,1
love,1
machine,1
of,1


## Manual TF-IDF Computation


## Comparing With `TfidfVectorizer`'s Output

The *IDF* values and *relative pattern* match exactly what `TfidfVectorizer` computes internally, but the final numbers won't be identical to `TfidfVectorizer().fit_transform(documents)`'s own output, because scikit-learn applies one more step by default: **L2 row normalization** — scaling each document's vector so its overall length (Euclidean norm) equals 1.

Concretely, for document 0 ("I love programming."):

| | `love` | `programming` |
|---|---|---|
| This manual version | 1.916 | 1.916 |
| `TfidfVectorizer`'s actual output | 0.707 | 0.707 |

Same *ratio* between the two words (they remain equal to each other), just scaled down — 1.916 divided by the row's overall vector length (≈2.71) gives 0.707.

**Why this normalization matters in practice:** it prevents **longer documents** (which naturally accumulate higher raw TF-IDF scores simply from having more words) from automatically appearing "more important" than short documents purely due to length. After normalization, every document's vector has the same overall magnitude, so comparisons between documents (e.g. via cosine similarity) reflect content, not length.

If exactly matching `TfidfVectorizer`'s numbers is the goal, an extra row-normalization step would need to be added:

```python
tfidf_manual_normalized = tfidf_manual / np.linalg.norm(tfidf_manual.values, axis=1, keepdims=True)
```

In [ ]:
import numpy as np

# Term Frequency (TF)
tf = df_bow.values


# Document Frequency (DF)
df = (df_bow > 0).sum(axis=0)


# Inverse Document Frequency (IDF)
idf = np.log((len(documents) + 1) / (df + 1)) + 1  # Adding 1 to avoid division by zero and following sklearn's formula
# Compute IDF for every word using scikit-learn's exact SMOOTHED
# formula (not quite the plain log(N/df) formula given at the start —
# this is the refined version, matching what TfidfVectorizer actually
# uses internally):
#   - len(documents) + 1: adds 1 to the total document count (N=4 -> 5)
#   - df + 1: adds 1 to each word's document frequency
#   - + 1 at the very end: adds 1 to the final log result
#   Together, these three "+1"s prevent two problems with the plain
#   formula: (a) division by zero if a word had df=0, and (b) a word
#   appearing in EVERY document getting an IDF of exactly log(1)=0,
#   which would zero out its TF-IDF score entirely regardless of how
#   often it appears — smoothing keeps even very common words at a
#   small positive IDF instead of exactly 0.

df = np.array(df)
idf = np.array(idf)
# Convert both from pandas Series to plain NumPy arrays, so the
# element-wise multiplication below behaves as pure array arithmetic
# rather than pandas' index-aware operations.

# TF-IDF
tfidf_manual = pd.DataFrame(tf * idf, columns=df_bow.columns)
# Multiply TF and IDF element-wise: `tf` is (4, 15), `idf` is (15,) —
# NumPy automatically BROADCASTS idf across all 4 rows, applying each
# word's single IDF value to that word's column in every document.
# This directly implements TF-IDF(t) = TF(t) x IDF(t) from the
# formula, wrapped back into a labeled DataFrame for readability.

tfidf_manual
# Display the resulting 4x15 table of manually-computed TF-IDF scores.


,an,data,field,interesting,is,language,learning,love,machine,of,programming,python,science,subset,versatile
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.916291,0.000000,0.000000,1.916291,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,1.223144,1.916291,0.000000,0.000000,0.000000,0.000000,0.000000,1.916291,0.000000,0.000000,1.916291
2,1.916291,1.510826,1.916291,1.916291,1.223144,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.510826,0.000000,0.000000
3,0.000000,1.510826,0.000000,0.000000,1.223144,0.000000,1.916291,0.000000,1.916291,1.916291,0.000000,0.000000,1.510826,1.916291,0.000000


Scikit-learn has a transformer class that performs both the BoW and TF-IDF for simplicity.

In [ ]:
vectorizer_tfidf = TfidfVectorizer(norm=None)  # Disable L2 normalization for comparison
# Create a TfidfVectorizer instance, but with norm=None — explicitly
# turning OFF the default L2 row-normalization flagged as the
# discrepancy in the previous cell. This is done specifically so this
# built-in version can be compared directly, apples-to-apples, against
# the manually-computed tfidf_manual from before, without the
# normalization step muddying the comparison.
# (By default, without this argument, TfidfVectorizer would apply
# norm='l2', which is what produced the 0.707-style scaled-down values
# seen in the very first TfidfVectorizer output earlier in this
# notebook.)

X_tfidf = vectorizer_tfidf.fit_transform(documents)
# Same fit_transform pattern as CountVectorizer: builds the vocabulary,
# computes TF, computes IDF (using the same smoothed formula
# implemented by hand in the previous cell), and multiplies them
# together — internally, in one call — across all 4 documents.

df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=vectorizer_tfidf.get_feature_names_out())
df_tfidf
# Convert to a dense, labeled DataFrame for display — same conversion
# pattern used for df_bow earlier.

,an,data,field,interesting,is,language,learning,love,machine,of,programming,python,science,subset,versatile
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.916291,0.000000,0.000000,1.916291,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,1.223144,1.916291,0.000000,0.000000,0.000000,0.000000,0.000000,1.916291,0.000000,0.000000,1.916291
2,1.916291,1.510826,1.916291,1.916291,1.223144,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.510826,0.000000,0.000000
3,0.000000,1.510826,0.000000,0.000000,1.223144,0.000000,1.916291,0.000000,1.916291,1.916291,0.000000,0.000000,1.510826,1.916291,0.000000


## Training models on the IMDB dataset

We use the IMDB movie reviews dataset for a binary classification task, where the goal is to classify movie reviews as either positive or negative. We apply both BoW and TF-IDF encoding techniques to the text data, train a Logistic Regression model, and evaluate the model's performance using a classification report. The classification report provides key metrics such as precision, recall, and F1-score, giving a comprehensive view of how well the model performs for each class (positive and negative reviews) under both encoding schemes.

Experimenting with different text encoding techniques is a crucial step in handling text classification problems, as the choice of encoding can significantly impact the model's performance.

In [ ]:
!pip install datasets
# Install Hugging Face's `datasets` library — a package that provides
# easy, one-line access to hundreds of common ML datasets (including
# IMDB), without needing to manually download, unzip, or parse files
# yourself. Distinct from scikit-learn's `sklearn.datasets` (used
# earlier for Iris, digits, breast cancer, Swiss roll) — those were
# small built-in toy datasets bundled directly with scikit-learn,
# whereas IMDB (50,000 movie reviews) is large enough that it needs to
# be fetched from Hugging Face's dataset hub instead.
#
# This sets up the transition flagged much earlier in this
# conversation (back when the IMDb dataset was first discussed as a
# case study for data types and visualization) — this is now the point
# where that dataset actually gets loaded and used for real, this time
# specifically for the text (review) column, encoded via BoW/TF-IDF,
# to train an actual sentiment classifier.

In [ ]:
from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
# Bringing back CountVectorizer (BoW), TfidfVectorizer, LogisticRegression,
# and classification_report — the exact same tools used on the toy
# 4-document corpus and the imbalanced synthetic dataset earlier in
# this series, now applied together on a large, real dataset.

# Load the IMDB dataset using Hugging Face's datasets library
dataset = load_dataset("stanfordnlp/imdb")
# Downloads (or loads from local cache, on repeat runs) the IMDB movie
# reviews dataset by its registered name on the Hugging Face Hub.
# Returns a DatasetDict — a dictionary-like object containing multiple
# named splits, similar in spirit to how sklearn's load_iris() Bunch
# object bundled data/target/feature_names together, but here it
# bundles multiple full DATA SPLITS (train/test/etc.) instead.

# The data is split into train, test, and unsupervised (which we won't use here)
# The IMDB dataset ships with three pre-defined splits:
#   - 'train': 25,000 labeled reviews (12,500 positive, 12,500 negative)
#   - 'test': 25,000 labeled reviews (12,500 positive, 12,500 negative)
#   - 'unsupervised': 50,000 additional reviews with NO labels — not
#     usable for supervised classification (hence the comment "which
#     we won't use here"), but included in the original dataset for
#     unsupervised/semi-supervised research (e.g. language model
#     pretraining on review text).
# Unlike the earlier synthetic imbalanced dataset, note IMDB's
# train/test splits are already perfectly balanced 50/50 between
# classes — the class-imbalance techniques from the previous notebook
# won't be needed here.

X_train, y_train = dataset['train']['text'], dataset['train']['label']
X_test, y_test = dataset['test']['text'], dataset['test']['label']
# Extract the raw review text and labels from each split:
#   - dataset['train']['text']: a list of 25,000 raw review strings
#     (the actual written movie reviews) — this is the RAW TEXT that
#     still needs to be encoded (via BoW or TF-IDF) before any model
#     can use it, exactly the problem set up by the whole text-encoding
#     discussion so far.
#   - dataset['train']['label']: a list of 25,000 integers, where
#     0 = negative review, 1 = positive review — encoded this way by
#     the dataset's creators, so no additional label-encoding step is
#     needed (unlike the raw text, which still needs vectorizing).
#   - Same extraction repeated for the 'test' split, kept as a
#     completely separate hold-out set (recall the hold-out
#     definition from earlier) for final, honest evaluation.

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
(len(X_train), len(X_test))

(25000, 25000)

In [ ]:
# Encoding and Model Training: Bag of Words
vectorizer_bow = CountVectorizer()
# Create a fresh CountVectorizer, same as with the toy 4-document
# corpus — but this time it will build its vocabulary from all 25,000
# real, full-length IMDB training reviews, so the resulting vocabulary
# will be vastly larger (likely tens of thousands of unique words,
# rather than 15).

X_train_bow = vectorizer_bow.fit_transform(X_train)
# fit_transform on the TRAINING text only: builds the vocabulary
# purely from the training reviews, then converts each training
# review into its BoW count vector.

X_test_bow = vectorizer_bow.transform(X_test)
# transform ONLY (no fit) on the test text — this is a critical
# distinction from the training line above, and it's the text-data
# equivalent of the "fit on training data, transform both train and
# test" principle flagged back when StandardScaler was introduced.
# Using .transform() here means the test reviews are converted into
# vectors using the EXACT SAME vocabulary learned from training data —
# any word that appears in a test review but was never seen in any
# training review is simply ignored (it has no corresponding column
# in the vocabulary, so it can't be counted).
# This matters a lot for correctness: if .fit_transform() were called
# on the test set instead, it would build a DIFFERENT vocabulary from
# test-only words, producing vectors with a different number of
# columns/meaning than the training vectors — the model wouldn't even
# be able to use them, since LogisticRegression expects test vectors
# to have the exact same feature dimensions it was trained on. Using
# .transform() (fit only once, on training data) is what guarantees
# that consistency, and also avoids "peeking" at test-set vocabulary
# during training — a form of data leakage.

model_bow = LogisticRegression(max_iter=1000)
# Create a logistic regression model. max_iter=1000 raises the
# iteration cap for the underlying optimizer (default is 100) — with
# a very large, sparse, high-dimensional feature space like BoW-encoded
# text (tens of thousands of columns), the solver often needs more
# iterations to converge than it would on small, low-dimensional data
# like the earlier Iris/imbalanced examples; without raising this,
# scikit-learn would likely print a ConvergenceWarning.

model_bow.fit(X_train_bow, y_train)
# Train logistic regression on the BoW-encoded training reviews and
# their labels (0 = negative, 1 = positive) — learning which words'
# presence/frequency are predictive of positive vs. negative sentiment.

y_pred_bow = model_bow.predict(X_test_bow)
# Predict sentiment labels for all 25,000 test reviews, using the
# vocabulary-consistent BoW vectors built above.

print(f'Classification Report (Bag of Words):\n{classification_report(y_test, y_pred_bow)}')
# Print precision/recall/F1/support per class, plus overall accuracy —
# same diagnostic tool used in the imbalance notebook, though here the
# classes are already balanced (12,500/12,500 in both train and test),
# so accuracy alone is a more trustworthy summary than it was there.

Classification Report (Bag of Words):
              precision    recall  f1-score   support

           0       0.86      0.88      0.87     12500
           1       0.87      0.86      0.87     12500

    accuracy                           0.87     25000
   macro avg       0.87      0.87      0.87     25000
weighted avg       0.87      0.87      0.87     25000



## Explaining the Output: Bag of Words Classification Report

Classification Report (Bag of Words):
precision recall f1-score support

       0       0.86      0.88      0.87     12500
       1       0.87      0.86      0.87     12500

accuracy                           0.87     25000

macro avg 0.87 0.87 0.87 25000
weighted avg 0.87 0.87 0.87 25000


### Reading the per-class rows

- **Class 0 (negative reviews):**
  - **Precision = 0.86** — of all reviews the model *predicted* as negative, 86% actually were negative (14% were false alarms — positive reviews mistakenly called negative).
  - **Recall = 0.88** — of all reviews that *actually were* negative, the model correctly caught 88% of them (12% of truly negative reviews were missed and predicted as positive).
  - **F1-score = 0.87** — the harmonic mean of precision and recall for this class, summarizing both into one balanced number.
  - **Support = 12,500** — there were 12,500 true negative reviews in the test set.

- **Class 1 (positive reviews):**
  - **Precision = 0.87**, **Recall = 0.86**, **F1-score = 0.87**, **Support = 12,500** — essentially the mirror image of class 0's numbers, which makes sense on a perfectly balanced dataset: the model's tendency to slightly favor recall on class 0 (0.88) corresponds to slightly lower recall on class 1 (0.86), since a review it correctly recalls as negative is one it correctly avoids mislabeling as positive.

### Reading the summary rows

- **Accuracy = 0.87** — overall, the model got 87% of all 25,000 test reviews right. Unlike the earlier imbalanced-dataset example (where 88% accuracy was hiding a completely broken minority class), this accuracy figure is **trustworthy here**, precisely *because* both classes are perfectly balanced (12,500 each) and — as shown above — both classes have similarly strong precision/recall. There's no hidden failure mode being masked.
- **Macro avg = 0.87 across the board** — the plain, unweighted average of the two classes' scores (treats both classes equally regardless of size). It matches accuracy closely here specifically because both classes are the same size and perform similarly well — this is what a *healthy*, non-imbalanced classification report looks like, in contrast to the wide accuracy/macro-avg split seen with the imbalanced dataset earlier.
- **Weighted avg = 0.87 across the board** — the average weighted by each class's support (sample count). With equal class sizes, this is mathematically identical to the macro average here — the distinction between macro and weighted avg only becomes meaningful when class sizes differ, as they did in the imbalance notebook.

### Overall takeaway

A **0.87 F1-score for both classes**, with all three summary metrics (accuracy, macro avg, weighted avg) landing at the same 0.87, indicates a genuinely well-performing, balanced classifier — not one class being favored at the expense of the other. This is a strong baseline result for BoW + logistic regression on IMDB sentiment: the model correctly identifies sentiment roughly 87% of the time, using nothing more sophisticated than raw word-count frequencies, with no understanding of word order, negation, or context (e.g. "not bad" vs. "bad").

In [ ]:
X_train_bow.shape

(25000, 74849)

In [ ]:
# Encoding and Model Training: TF-IDF
vectorizer_tfidf = TfidfVectorizer()
# Create a fresh TfidfVectorizer with default settings — this time
# WITHOUT norm=None, so the default L2 row-normalization from earlier
# is active. Same vocabulary-building principle as CountVectorizer,
# but each review's vector will now be weighted by TF x IDF (down-
# weighting common words like "the," "is," "movie" that appear in
# nearly every review, up-weighting distinctive words) and then
# normalized to unit length.

X_train_tfidf = vectorizer_tfidf.fit_transform(X_train)
# fit_transform on the training text: builds the vocabulary AND learns
# each word's IDF (based on how many of the 25,000 training reviews
# contain it), then converts each training review into its TF-IDF
# vector.

X_test_tfidf = vectorizer_tfidf.transform(X_test)
# transform ONLY on the test text — same principle as the BoW cell:
# reuse the vocabulary AND the IDF values learned from training data,
# rather than recomputing them from the test set. This keeps the
# comparison with the BoW model fair (both models see test data
# processed consistently with how they were trained) and avoids
# leaking test-set word-frequency information into the encoding.

model_tfidf = LogisticRegression(max_iter=1000)
model_tfidf.fit(X_train_tfidf, y_train)
# Train a separate logistic regression model (a new instance,
# model_tfidf, distinct from model_bow) on the TF-IDF vectors — same
# max_iter=1000 raised iteration cap, same reasoning as before given
# the large, high-dimensional, sparse feature space.

y_pred_tfidf = model_tfidf.predict(X_test_tfidf)
print(f'Classification Report (TF-IDF):\n{classification_report(y_test, y_pred_tfidf)}')
# Predict and report performance on the same 25,000-review test set
# used for the BoW model, so the two classification reports are
# directly comparable — isolating text encoding method as the only
# variable that changed between the two runs.


Classification Report (TF-IDF):
              precision    recall  f1-score   support

           0       0.88      0.88      0.88     12500
           1       0.88      0.88      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000



## Explaining the Output: TF-IDF Classification Report

Classification Report (TF-IDF):
precision recall f1-score support

       0       0.88      0.88      0.88     12500
       1       0.88      0.88      0.88     12500

accuracy                           0.88     25000

macro avg 0.88 0.88 0.88 25000
weighted avg 0.88 0.88 0.88 25000


### Reading the per-class rows

- **Class 0 (negative reviews):** precision = 0.88, recall = 0.88, F1 = 0.88, support = 12,500.
- **Class 1 (positive reviews):** precision = 0.88, recall = 0.88, F1 = 0.88, support = 12,500.

Every single number in this report — both classes' precision, recall, and F1, plus accuracy, macro avg, and weighted avg — comes out to exactly **0.88**. This near-perfect symmetry between the two classes shows the model isn't favoring positive over negative (or vice versa) in either its precision or its recall — a genuinely balanced classifier, just like the BoW result before it, but slightly stronger across the board.

### Direct comparison with the Bag of Words result

| Metric | BoW | TF-IDF | Change |
|---|---|---|---|
| Class 0 F1 | 0.87 | 0.88 | +0.01 |
| Class 1 F1 | 0.87 | 0.88 | +0.01 |
| Accuracy | 0.87 | 0.88 | **+0.01** |
| Macro avg F1 | 0.87 | 0.88 | +0.01 |

### What this confirms

TF-IDF gives a **small but consistent improvement** over plain BoW — exactly the modest gain predicted earlier, rather than a dramatic one. This lines up with the reasoning laid out before running it: sentiment classification is a task where strongly opinion-laden words ("terrible," "brilliant," "waste," "masterpiece") already stand out clearly under raw BoW counts, since they rarely appear in *neutral* filler text. TF-IDF's main advantage — down-weighting common, low-information words like "movie," "film," or "the" — helps at the margins by slightly sharpening the signal, but it isn't fixing a major weakness the way it might on a task where distinguishing documents relies more heavily on rarer, topic-specific vocabulary (e.g. document retrieval or topic classification, rather than sentiment).

**Practical takeaway:** for this specific problem — IMDB sentiment classification with logistic regression — TF-IDF is a safe, essentially "free" upgrade over BoW (same code structure, same training time order of magnitude, no extra hyperparameters to tune), consistently edging out BoW by about a percentage point across every metric, without introducing any new trade-offs like the ones seen in the imbalance notebook (where the under/over-sampling gains in recall came at the cost of precision and accuracy). This is a case where the "better" encoding choice is a clean win, not a trade-off.